# HASA: heterogeneity-aware subnet allocation

**HASA: Subnet Allocation for Compute-Constrained Model-Heterogeneous Federated Learning**
Amir Hossein Shahdadian, Ahmed M. Abdelmoniem, Mahdi Taheri, Samira Nazari, Christian Herglotz · IEEE EDGE 2026
[Paper](https://arxiv.org/abs/2606.07621) · [Code](https://github.com/Reliable-and-Secure-Systems/HASA)

In model-heterogeneous federated learning, every client trains a subnet of one shared model, sized to what the client can afford. HASA decides how wide each subnet should be. It measures how far each client's training data is from the data of the whole federation, gives the wider subnets to the clients that are most different, and keeps the total compute budget fixed.

This notebook runs the article-title experiment of the paper from start to finish:

1. set up the federation: seven Medium publications, one client each;
2. score how different each client's data is;
3. turn the scores into subnet widths under a fixed budget;
4. see what a width means inside the model;
5. train the federated model with HASA widths and with uniform widths, over the paper's ten seeds;
6. compare the two, overall and client by client.

The first steps take seconds. Training is the slow part: twenty federated runs of a few minutes each on a GPU, so choose a GPU runtime in Colab (*Runtime → Change runtime type → T4 GPU*).

## 1. Setup

In [ ]:
import os
if os.path.exists("../hasa"):          # opened from notebooks/ inside the repository
    os.chdir("..")
elif not os.path.exists("hasa"):       # Google Colab: fetch the code
    !git clone -q https://github.com/Reliable-and-Secure-Systems/HASA.git
    os.chdir("HASA")
!pip -q install --disable-pip-version-check demoji==1.1.0

import time
import numpy as np, pandas as pd
from hasa import load_benchmark, heterogeneity_scores, hasa_allocation, run_policy, get_device, device_name
from hasa.report import summary_table, plot_results, per_client_table, plot_client_gains

device = get_device()
print("running on", device_name(device))

## 2. Seven publications, seven clients

Each client is one Medium publication. Its articles are split 70/10/20 into training, validation and test articles, and every title becomes a set of next-word prediction examples. The clients are very unequal in size, as in most real federations.

In [ ]:
bench = load_benchmark("data/medium_data.csv", verbose=False)

clients = pd.DataFrame({"articles": [c.n_articles for c in bench.clients],
                        "training examples": bench.client_data_sizes,
                        "test examples": [c.sizes["test"] for c in bench.clients]},
                       index=pd.Index(bench.client_names, name="client"))
clients.style.bar(subset=["training examples"], color="#E9C9D5")

## 3. How different is each client?

HASA compares the word distribution of each client's training titles with the pooled distribution of the whole federation, using the Jensen-Shannon divergence. Only training data is used. A higher score means the client's data is further from the mix.

In [ ]:
scores = heterogeneity_scores(bench)
clients["heterogeneity (JSD)"] = scores

(clients[["training examples", "heterogeneity (JSD)"]].style
    .bar(subset=["heterogeneity (JSD)"], color="#E9C9D5")
    .format({"heterogeneity (JSD)": "{:.3f}"}))

## 4. From scores to subnet widths

This is Algorithm 1 of the paper. The scores are ranked, and the ranks are mapped linearly to widths between 0.2 and 0.8. The widths are then rescaled so that their size-weighted mean equals the compute budget B = 0.5, and clipped to the bounds. Uniform allocation gives every client 0.5, so both allocations cost the same compute.

In [ ]:
n = np.array(bench.client_data_sizes, dtype=float)
p = n / n.sum()                                                     # client weights n_i / N
r = 0.2 + 0.6 * np.argsort(np.argsort(scores)) / (len(scores) - 1)  # rank, then map to [0.2, 0.8]
for _ in range(2):
    r = np.clip(r * 0.5 / (p @ r), 0.2, 0.8)                        # meet the budget, stay in bounds
assert np.allclose(r, hasa_allocation(bench.client_data_sizes, scores))   # same as the library

clients["HASA width (%)"] = 100 * r
print(f"size-weighted mean width: HASA {p @ r:.3f}, uniform 0.500 (budget 0.5)")
(clients[["training examples", "heterogeneity (JSD)", "HASA width (%)"]].style
    .bar(subset=["HASA width (%)"], color="#E9C9D5", vmin=0, vmax=100)
    .format({"heterogeneity (JSD)": "{:.3f}", "HASA width (%)": "{:.1f}"}))

## 5. What a width means inside the model

All clients share one LSTM with 128-dimensional embeddings and 256 hidden units. A client with width r trains only the first ⌊256·r⌋ hidden units. The function below cuts those units out of the shared weights (their gate rows, their recurrent weights and their output columns) and runs the LSTM on the slices, so a narrow subnet really computes less.

In [ ]:
import inspect
from hasa.model import forward_subnet_with_indices
print(inspect.getsource(forward_subnet_with_indices))

units = [max(16, int(256 * w)) for w in r]
clients["hidden units"] = units
clients["LSTM compute (%)"] = [100 * (4*a*128 + 4*a*a) / (4*256*128 + 4*256*256) for a in units]
clients[["HASA width (%)", "hidden units", "LSTM compute (%)"]].style.format("{:.1f}", subset=["HASA width (%)", "LSTM compute (%)"])

## 6. Federated training

For every seed, we train twice from the same initial model on the same data: once with uniform widths and once with the HASA widths. The protocol is the paper's: 50 rounds of FedAvg with all seven clients, and one local epoch of Adam (learning rate 0.001, batch size 64) per round. After the last round, every client is scored with the global model on its own test set, at its own width. We repeat this for the ten seeds used in the paper.

In [ ]:
SEEDS = [42, 123, 456, 789, 1011, 2022, 3033, 4044, 5055, 6066]
POLICIES = {"Uniform": "uniform_50", "HASA": "difficulty_only"}

runs = {name: [] for name in POLICIES}
start = time.time()
for seed in SEEDS:
    for name, policy in POLICIES.items():
        h = run_policy(bench, scores, policy, device, seed=seed, log=None)
        runs[name].append(h)
        print(f"seed {seed:>4} · {name:<7}  mean {100*h['test_mean'][-1]:5.2f}%  "
              f"worst {100*h['test_worst'][-1]:5.2f}%  P10 {100*h['test_p10'][-1]:5.2f}%")
print(f"done in {(time.time() - start) / 60:.0f} minutes")

## 7. Results

We look at three client-level numbers: the mean accuracy over the clients, the accuracy of the worst client, and the 10th percentile (P10). The table gives the mean and standard deviation over the seeds, the average paired change, and a one-sided paired t-test across seeds.

In [ ]:
display(summary_table(runs))
plot_results(runs, SEEDS)

## 8. Which clients gain?

The same comparison for every client, averaged over the seeds.

In [ ]:
per_client = per_client_table(runs, bench.client_names, r)
display(per_client)
plot_client_gains(per_client)

## Try it yourself

- Flip the direction and give the wide subnets to the least heterogeneous clients: `run_policy(bench, scores, "difficulty_only", device, seed=42, invert_difficulty=True)`.
- Allocate by data size instead (`"size_only"`), or by a mix of size and heterogeneity (`"mixed"`).
- Every run keeps its history every fifth round, for example `runs["HASA"][0]["test_mean"]`.